# Physics-Informed Neural Network Example

This notebook contains the same worked example that is embedded in `pinn.qmd`.

This example briefly restates the main objects from the slides before showing the code, so it can be read mostly on its own.

As in the slides, we write the network prediction as ${\rm ANN}(t)$. For the RC circuit,

$$
\tau\,\dot V_C(t)+V_C(t)-V_{\mathrm{in}}(t)=0.
$$

This is the \textbf{physics error}. Replacing the unknown voltage by the network prediction gives

$$
r(t)=\tau\,\dot {\rm ANN}(t)+{\rm ANN}(t)-V_{\mathrm{in}}(t).
$$

We then separate the data term and the physics term:

$$
\mathcal{L}_{\rm data}
=
\frac{1}{N}\sum_{i=1}^{N}\left(V_i-{\rm ANN}(t_i)\right)^2
\qquad
\mathcal{L}_{\rm phys}
=
\frac{1}{M}\sum_{j=1}^{M}r(\tau_j)^2
$$

and combine them into the full PINN loss

$$
\mathcal{L}
=
\lambda_{\rm data}\,\mathcal{L}_{\rm data}
+
\lambda_{\rm phys}\,\mathcal{L}_{\rm phys}.
$$

We call this a \textbf{physics-informed neural network (PINN)}.\footnote{M.~Raissi, P.~Perdikaris, and G.~E.~Karniadakis, ``Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations,'' \textit{Journal of Computational Physics}, 2019.}

Here, $t_i$ are the measurement times, $V_i$ are the measured voltages, and $\tau_j$ are the physics check points chosen from the physics interval.

So the walkthrough follows the same logic as the NN example, with one extra ingredient:

1. define the experiment and the available data,
2. define the model ${\rm ANN}(t)$,
3. define the physics error $r(t)$, the terms $\mathcal{L}_{\rm data}$ and $\mathcal{L}_{\rm phys}$, and the full loss $\mathcal{L}$,
4. write one Adam update for the trainable weights and biases,
5. repeat that update and inspect the result.

At the top, we collect a few parameters that you can vary without touching the rest of the example.

In [ ]:
num_data_points = 8
num_residual_points = 160
data_interval = (0.00, 0.12)
physics_interval = (0.12, 0.45)
noise_level = 0.03
lambda_phys = 1.0
layer_widths = [1, 96, 96, 1]
num_training_steps = 1500

These eight choices control the example:

- `num_data_points`: the number of noisy measurement pairs $(t_i,V_i)$ used in the data term,
- `num_residual_points`: the number of physics check points $\tau_j$ used in the physics term,
- `data_interval`: the time window from which the measurement times $t_i$ are sampled,
- `physics_interval`: the time window from which the physics check points $\tau_j$ are sampled,
- `noise_level`: the size of the random perturbation added to the exact voltages,
- `lambda_phys`: the weight of the physics term in the total PINN loss,
- `layer_widths`: the widths of the successive layers of the neural network,
- `num_training_steps`: the number of Adam updates performed during training.

For example, `layer_widths = [1, 96, 96, 1]` means one scalar input time, two hidden layers of width `96`, and one scalar output voltage.

If you change `layer_widths`, keep the first and last entry equal to `1`, because both the input $t$ and the output $V_C(t)$ are one-dimensional in this example.

By default, the measurements are taken only on the early charging interval, while the physics check points extend through the later part of the experiment.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

**Step 1. Define the experiment and the available data.** The switching experiment uses the same RC circuit as in the slides: the capacitor first charges, then the input is switched off and the capacitor discharges.

In [ ]:
R, C = 1.0e3, 100.0e-6
tau = R * C
t_end, t_switch = 0.45, 0.15
t_plot = jnp.linspace(0.0, t_end, 300)
data_start, data_end = data_interval
phys_start, phys_end = physics_interval

if not (0.0 <= data_start < data_end <= t_end):
    raise ValueError("data_interval must satisfy 0 <= start < end <= t_end.")

if not (0.0 <= phys_start < phys_end <= t_end):
    raise ValueError("physics_interval must satisfy 0 <= start < end <= t_end.")


def V_in(t):
    t = jnp.asarray(t)
    return jnp.where(t < t_switch, 5.0, 0.0)


def V_true(t):
    t = jnp.asarray(t)
    V_switch = 5.0 * (1.0 - jnp.exp(-t_switch / tau))
    charge = 5.0 * (1.0 - jnp.exp(-t / tau))
    discharge = V_switch * jnp.exp(-(t - t_switch) / tau)
    return jnp.where(t < t_switch, charge, discharge)


t_data = jnp.linspace(data_start, data_end, num_data_points)
tau_phys = jnp.linspace(phys_start, phys_end, num_residual_points)
measurement_key = jax.random.PRNGKey(31)
V_data = V_true(t_data) + noise_level * jax.random.normal(
    measurement_key, t_data.shape
)

Here:

- `t_data` contains the measurement times $t_i$,
- `V_data` contains the measured voltages $V_i$,
- `tau_phys` contains the physics check points $\tau_j$.

The opening plot makes the task visually clear.

In [ ]:
fig, ax = plt.subplots(figsize=(6.3, 3.4))

ax.plot(t_plot, V_true(t_plot), color="black", linewidth=2.0, label="Reference Curve")
ax.scatter(t_data, V_data, color="#c44e52", s=26, zorder=5, label="Noisy Measurements")
ax.axvspan(float(data_start), float(data_end), color="#f6c8cf", alpha=0.22, label="data interval")
ax.axvspan(float(phys_start), float(phys_end), color="#cfe8ff", alpha=0.28, label="physics interval")
ax.axvline(float(t_switch), color="#ff7f0e", linewidth=1.0, linestyle="-.", alpha=0.9)
ax.set_xlim(0.0, float(t_end))
ax.set_ylim(-0.35, 5.6)
ax.set_xlabel(r"$t$ [s]")
ax.set_ylabel(r"$V_C(t)$ [V]")
ax.set_title("Sparse Measurements on the Switching Example", fontsize=10)
ax.grid(True, linestyle=":", alpha=0.6)
ax.text(float(t_switch) + 0.008, 5.05, "Switch Off", color="#cc5500", fontsize=8.5)
ax.legend(frameon=True, fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

With the default intervals, the data stop before the switch-off and before the discharge phase, so the model must infer the later behavior from physics.

**Step 2. Define the model ${\rm ANN}(t)$.** As in the NN section, the network is built from repeated affine maps, that is, linear maps together with a bias term, followed by ReLU activations. If `layer_widths = [1, m, 1]`, then the model has the familiar one-hidden-layer form

$$
{\rm ANN}(t)=W_2\,\sigma(W_1 \tilde t+b_1)+b_2,
\qquad
\tilde t=\frac{2t}{t_{\mathrm{end}}}-1.
$$

The code below implements exactly that rule layer by layer, so deeper architectures can also be used just by changing `layer_widths`.

In [ ]:
def scale_time(t):
    return 2.0 * jnp.asarray(t).reshape(-1, 1) / t_end - 1.0


def ann_predict(network_parameters, t):
    activations = scale_time(t)

    for layer in network_parameters[:-1]:
        activations = jax.nn.relu(
            activations @ layer["weights"] + layer["bias"]
        )

    output_layer = network_parameters[-1]
    return (activations @ output_layer["weights"] + output_layer["bias"])[:, 0]

So the visible code matches the notation directly:

- `network_parameters` stores all trainable weights and biases,
- `ann_predict(network_parameters, t)` is the code version of ${\rm ANN}(t)$,
- `scale_time(t)` creates the rescaled input $\tilde t$ and arranges the times as an `(N, 1)` array, so the network sees one scalar feature per sample.

The folded helper below only initializes the trainable parameters.

In [ ]:
def initialize_network_parameters(layer_widths, key):
    layer_keys = jax.random.split(key, len(layer_widths) - 1)
    network_parameters = []

    for layer_key, input_width, output_width in zip(
        layer_keys, layer_widths[:-1], layer_widths[1:]
    ):
        network_parameters.append(
            {
                "weights": jax.random.normal(layer_key, (input_width, output_width))
                * jnp.sqrt(2.0 / input_width),
                "bias": jnp.zeros(output_width),
            }
        )

    return tuple(network_parameters)

**Step 3. Define the physics error and the PINN loss.** Now we add the PINN-specific ingredient. The physics error is the direct code translation of the RC equation.

In [ ]:
def ann_predict_scalar(network_parameters, t):
    return ann_predict(network_parameters, t)[0]


ann_time_derivative = jax.grad(ann_predict_scalar, argnums=1)


def physics_error(network_parameters, t):
    t = jnp.asarray(t)
    return jax.vmap(
        lambda time: (
            tau * ann_time_derivative(network_parameters, time)
            + ann_predict_scalar(network_parameters, time)
            - V_in(time)
        )
    )(t)

This is the one-to-one correspondence with the mathematics:

- `ann_predict_scalar(...)` is the scalar version of ${\rm ANN}(t)$,
- `ann_time_derivative` is the derivative $\dot {\rm ANN}(t)$,
- `physics_error(network_parameters, t)` is the physics error $r(t)$.

Now we translate the loss formula into code.

In [ ]:
lambda_data = 1.0


def mean_squared_error(values, targets):
    return jnp.mean((values - targets) ** 2)


def data_loss(network_parameters):
    return mean_squared_error(ann_predict(network_parameters, t_data), V_data)


def physics_loss(network_parameters):
    return mean_squared_error(physics_error(network_parameters, tau_phys), 0.0)


def loss_values(network_parameters):
    data_value = data_loss(network_parameters)
    physics_value = physics_loss(network_parameters)
    total_value = lambda_data * data_value + lambda_phys * physics_value
    return data_value, physics_value, total_value


def total_loss(network_parameters):
    _, _, total_value = loss_values(network_parameters)
    return total_value

So:

- `data_loss(network_parameters)` is the code version of the data term
  $$
  \mathcal{L}_{\rm data}
  =
  \frac{1}{N}\sum_{i=1}^{N}\left(V_i-{\rm ANN}(t_i)\right)^2,
  $$
- `physics_loss(network_parameters)` is the code version of the physics term
  $$
  \mathcal{L}_{\rm phys}
  =
  \frac{1}{M}\sum_{j=1}^{M}r(\tau_j)^2,
  $$
- `lambda_data` and `lambda_phys` are the code versions of $\lambda_{\rm data}$ and $\lambda_{\rm phys}$,
- `total_loss(network_parameters)` is the full PINN loss $\mathcal{L}$.

**Step 4. Write one Adam update.** The training step has exactly the same structure as in the earlier NN example. The only difference is that the loss now contains both data and physics.

In [ ]:
network_parameters = initialize_network_parameters(layer_widths, jax.random.PRNGKey(21))
learning_rate = 3.0e-3
optimizer = optax.adam(learning_rate)
opt_state = optimizer.init(network_parameters)


@jax.jit
def train_step(network_parameters, opt_state):
    loss_value, gradients = jax.value_and_grad(total_loss)(network_parameters)
    updates, opt_state = optimizer.update(gradients, opt_state, network_parameters)
    network_parameters = optax.apply_updates(network_parameters, updates)
    return network_parameters, opt_state, loss_value

This is the code form of one update of the trainable weights and biases, except that Adam chooses a smarter step than plain steepest descent.

**Step 5. Repeat the update and inspect the result.**

In [ ]:
report_every = 500

for step in range(num_training_steps):
    network_parameters, opt_state, _ = train_step(network_parameters, opt_state)

    if (step + 1) % report_every == 0:
        data_value, physics_value, loss_value = loss_values(network_parameters)
        print(
            f"step {step + 1:4d}: "
            f"data = {float(data_value):.4e}, "
            f"physics = {float(physics_value):.4e}, "
            f"total = {float(loss_value):.4e}"
        )

The optimizer therefore keeps updating the trainable weights and biases so that the model improves both on the measured data and on the RC equation.

In this example, the predicted voltage curve already becomes quite accurate before the total PINN loss has fully settled. That is not a contradiction: Adam does not force the loss to decrease monotonically, and the physics term is sensitive to local slope mismatches, especially near the switching time. So a visually very good prediction can still come with some oscillation in the reported loss values.

The final plot shows all ingredients together: the measurements, the physics check points, the Reference Curve, and the PINN Prediction.

In [ ]:
predicted_voltage = ann_predict(network_parameters, t_plot)
physics_point_y = jnp.full_like(tau_phys, -0.10)

fig, ax = plt.subplots(figsize=(6.3, 3.4))

ax.plot(t_plot, V_true(t_plot), color="black", linewidth=2.0, label="Reference Curve")
ax.plot(t_plot, predicted_voltage, color="#2ca02c", linewidth=2.1, label="PINN Prediction")
ax.scatter(t_data, V_data, color="#c44e52", s=26, zorder=5, label="Noisy Measurements")
ax.scatter(
    tau_phys,
    physics_point_y,
    s=14,
    color="#2ca02c",
    edgecolors="white",
    linewidths=0.4,
    alpha=0.85,
    zorder=6,
    label="Physics Check Points",
)

ax.axvspan(float(data_start), float(data_end), color="#f6c8cf", alpha=0.16)
ax.axvspan(float(phys_start), float(phys_end), color="#cfe8ff", alpha=0.22)
ax.axvline(float(t_switch), color="#ff7f0e", linewidth=1.0, linestyle="-.", alpha=0.9)

ax.set_xlim(0.0, float(t_end))
ax.set_ylim(-0.35, 5.6)
ax.set_xlabel(r"$t$ [s]")
ax.set_ylabel(r"$V_C(t)$ [V]")
ax.set_title("PINN Prediction on the Switching Example", fontsize=10)
ax.grid(True, linestyle=":", alpha=0.6)
ax.text(float(t_switch) + 0.008, 5.05, "Switch Off", color="#cc5500", fontsize=8.5)
ax.legend(frameon=True, fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

So the whole PINN loop can now be read directly in the notation of the slides:

1. define the model ${\rm ANN}(t)$,
2. define the physics error $r(t)$,
3. build the PINN loss $\mathcal{L}$ from data and physics,
4. update the weights and biases repeatedly,
5. inspect the resulting prediction ${\rm ANN}(t)$.

That is exactly ordinary neural-network training plus one extra source of information: the differential equation enters directly through the physics term in the loss.

**Further Tests**

- Reduce the amount or quality of data by using fewer measurement points, shortening `data_interval`, and/or increasing `noise_level`.
- Change the physics information by varying `num_residual_points` or moving `physics_interval` deeper into the no-data region.
- Compare a smaller network such as `layer_widths = [1, 24, 1]` with wider or deeper models.
- Vary `lambda_data`, `lambda_phys`, and `num_training_steps` to see how the balance between data fitting, physics enforcement, and optimization time affects the final prediction.
